# RAG con ChromaDB, agente ReAct, LangSmith y RAGAS (Simón Bolívar)

**Qué pide la tarea y qué hace este notebook**

1. **RAG**: textos sobre Simón Bolívar → embeddings → **Chroma** (BD vectorial) → recuperación por similitud → respuesta del LLM con ese contexto.
2. **Agente ReAct**: el modelo decide cuándo llamar a la herramienta que consulta Chroma (`create_agent` / fallback `create_react_agent`).
3. **LangSmith**: trazas con `LANGCHAIN_API_KEY` + `LANGCHAIN_TRACING_V2` (no es la misma clave que `OPENAI_API_KEY`).
4. **RAGAS**: al menos dos preguntas evaluadas con `faithfulness`, `answer_relevancy`, `context_precision`.
5. **Ejemplo obligatorio**: la pregunta *«¿Quién es Simón Bolívar?»* se resuelve con el pipeline (embedding de la consulta → búsqueda en Chroma → respuesta).

Los embeddings usan **`text-embedding-3-small`** por API de OpenAI (misma `OPENAI_API_KEY` que el chat). Si el enunciado exige *MiniLM local*, usa un venv aparte con `sentence-transformers`.


## 1. Dependencias

Instala paquetes ligeros (sin SciPy ni `sentence-transformers`). Lo ideal es un **venv** y `pip install -r requirements.txt` en la carpeta del proyecto; la celda siguiente también instala con el mismo Python del kernel.

**Reinicia el kernel** después de esta celda si antes tenías errores de imports.


In [1]:
import subprocess
import sys
from pathlib import Path

req = Path.cwd() / "requirements.txt"
cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "chromadb",
    "python-dotenv",
    "langchain",
    "langchain-openai",
    "langchain-chroma",
    "langchain-community",
    "langgraph",
    "datasets",
    "ragas",
    "ipywidgets",
]
if req.is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
else:
    subprocess.check_call(cmd)

importlib_ok = True
try:
    import chromadb  # noqa: F401
    from langchain_chroma import Chroma  # noqa: F401
    from langchain_openai import OpenAIEmbeddings  # noqa: F401
except Exception as e:
    importlib_ok = False
    print("Error importando:", e)

if importlib_ok:
    print("Dependencias OK. Intérprete:", sys.executable)


Dependencias OK. Intérprete: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe


## 2. Variables desde `.env`

- **`OPENAI_API_KEY`**: cuenta de OpenAI (chat + embeddings).  
- **`LANGCHAIN_API_KEY`**: cuenta de **LangSmith** (trazas); la generas en [smith.langchain.com](https://smith.langchain.com/settings). **No** es la clave de OpenAI.

Copia `.env.example` a `.env` y rellena ambas.


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Carga .env desde el directorio de trabajo actual (en VS Code/Cursor suele ser la carpeta del proyecto)
_env_path = Path.cwd() / ".env"
loaded = load_dotenv(_env_path, override=False)
if not _env_path.is_file():
    print(f"Aviso: no existe {_env_path.resolve()} — crea el archivo o exporta las variables en el sistema.")
elif not loaded:
    print(f"Aviso: { _env_path } existe pero load_dotenv no aplicó cambios (quizá ya estaban en el entorno).")

# Valores por defecto si no están en .env
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "ia2-rag-chromadb")

_required = ("OPENAI_API_KEY", "LANGCHAIN_API_KEY")
_missing = [k for k in _required if not os.environ.get(k)]
if _missing:
    print("Faltan variables obligatorias:", ", ".join(_missing))
else:
    print("Variables OK: OPENAI_API_KEY y LANGCHAIN_API_KEY están definidas.")


Variables OK: OPENAI_API_KEY y LANGCHAIN_API_KEY están definidas.


## 3. Base de conocimiento (Simón Bolívar)

Hechos cortos **hardcodeados** como lista de strings: el retriever convierte la pregunta en embeddings, busca en Chroma y el agente arma la respuesta con esos fragmentos.


In [3]:
documents = [
    "Simón Bolívar nació el 24 de julio de 1783 en Caracas, entonces parte del Virreinato de Nueva Granada.",
    "Es conocido como El Libertador por su papel militar y político en la independencia de Venezuela, Colombia, Ecuador, Perú y Bolivia frente al imperio español.",
    "En 1819 lideró la campaña que culminó en la batalla de Boyacá, decisiva para la independencia de la Nueva Granada.",
    "En 1819 pronunció el Discurso de Angostura, donde expuso ideas sobre república y gobierno para la América hispana.",
    "Fue presidente de Bolivia en 1825 y propulsó la creación de la República de Bolivia como estado soberano.",
    "Presidió la Gran Colombia, estado que unió gran parte del norte de Sudamérica entre 1819 y 1831.",
    "Murió el 17 de diciembre de 1830 en Santa Marta (actual Colombia); su legado simboliza la unidad latinoamericana.",
]


## 4. Chroma persistente + embeddings OpenAI

Se usa **`text-embedding-3-small`** (misma `OPENAI_API_KEY` que el chat). Los vectores se guardan en **`chroma_db_bolivar/`** (carpeta dedicada a esta base; si editas los textos, borra la carpeta y vuelve a ejecutar la celda para reindexar).


In [4]:
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

CHROMA_DIR = "./chroma_db_bolivar"
EMBED_MODEL = "text-embedding-3-small"

embedding_fn = OpenAIEmbeddings(model=EMBED_MODEL)
lc_docs = [Document(page_content=t) for t in documents]
persist = Path(CHROMA_DIR)

if persist.exists() and any(persist.iterdir()):
    print(f"[Chroma] Cargando {CHROMA_DIR} …")
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embedding_fn,
    )
    meta = vectorstore.get()
    count = len(meta.get("ids", []))
    if count > 0:
        print(f"[Chroma] {count} documentos en índice.")
    else:
        print("[Chroma] Carpeta sin datos; indexando…")
        vectorstore = Chroma.from_documents(
            documents=lc_docs,
            embedding=embedding_fn,
            persist_directory=CHROMA_DIR,
        )
        print("[Chroma] Listo:", len(lc_docs), "documentos.")
else:
    persist.mkdir(parents=True, exist_ok=True)
    vectorstore = Chroma.from_documents(
        documents=lc_docs,
        embedding=embedding_fn,
        persist_directory=CHROMA_DIR,
    )
    print("[Chroma] Indexados", len(lc_docs), "documentos.")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


[Chroma] Cargando ./chroma_db_bolivar …
[Chroma] 7 documentos en índice.


## 5. Agente ReAct con herramienta de recuperación

El modelo **gpt-4o-mini** decide cuándo consultar Chroma (comportamiento tipo ReAct). Con `LANGCHAIN_TRACING_V2=true` las ejecuciones deberían verse en LangSmith. La celda siguiente incluye la pregunta **«¿Quién es Simón Bolívar?»** para mostrar el pipeline completo.


In [5]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import create_retriever_tool
from langchain_openai import ChatOpenAI

retriever_tool = create_retriever_tool(
    retriever,
    name="buscar_en_base_vectorial",
    description="Busca hechos sobre la vida, batallas y legado de Simón Bolívar en la base vectorial.",
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LangGraph 1.x depreca create_react_agent; preferimos create_agent si está disponible.
try:
    from langchain.agents import create_agent

    agent = create_agent(
        model=llm,
        tools=[retriever_tool],
        system_prompt=(
            "Respondes en español sobre historia latinoamericana. "
            "Para datos sobre Simón Bolívar usa siempre la herramienta de búsqueda antes de afirmar fechas o hechos."
        ),
    )
except ImportError:
    from langgraph.prebuilt import create_react_agent

    agent = create_react_agent(llm, [retriever_tool])


def ejecutar_agente(pregunta: str) -> str:
    resultado = agent.invoke({"messages": [HumanMessage(content=pregunta)]})
    return resultado["messages"][-1].content


# Pregunta de ejemplo exigida por la tarea: embeddings de la consulta → Chroma → respuesta
pregunta_demostracion = "¿Quién es Simón Bolívar?"
print("Pregunta:", pregunta_demostracion)
print("Respuesta:", ejecutar_agente(pregunta_demostracion))


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Pregunta: ¿Quién es Simón Bolívar?
Respuesta: Simón Bolívar, conocido como "El Libertador", nació el 24 de julio de 1783 en Caracas, que en ese momento formaba parte del Virreinato de Nueva Granada. Es una figura clave en la historia de América Latina debido a su papel en la independencia de varios países, incluyendo Venezuela, Colombia, Ecuador, Perú y Bolivia, del dominio español.

Bolívar lideró importantes campañas militares y políticas, siendo la batalla de Boyacá en 1819 una de las más decisivas para la independencia de la Nueva Granada. Su legado perdura en la historia de la región, donde es recordado como un símbolo de libertad y lucha contra la opresión.


## 6. Al menos dos preguntas y contextos para RAGAS

Se recuperan contextos con el mismo retriever que usa el agente para alimentar las métricas.


In [6]:
preguntas_eval = [
    pregunta_demostracion,
    "¿Qué importancia tuvo la batalla de Boyacá en la carrera de Simón Bolívar?",
]

registros = []
for q in preguntas_eval:
    docs = retriever.invoke(q)
    ctx_docs = [
        d.page_content if hasattr(d, "page_content") else str(d) for d in docs
    ]
    respuesta = ejecutar_agente(q)
    registros.append({"question": q, "answer": respuesta, "contexts": ctx_docs})
    print("---")
    print("P:", q)
    print("Contextos:", ctx_docs)
    print("R:", respuesta[:500])


---
P: ¿Quién es Simón Bolívar?
Contextos: ['Simón Bolívar nació el 24 de julio de 1783 en Caracas, entonces parte del Virreinato de Nueva Granada.', 'Es conocido como El Libertador por su papel militar y político en la independencia de Venezuela, Colombia, Ecuador, Perú y Bolivia frente al imperio español.', 'En 1819 lideró la campaña que culminó en la batalla de Boyacá, decisiva para la independencia de la Nueva Granada.']
R: Simón Bolívar, conocido como "El Libertador", nació el 24 de julio de 1783 en Caracas, que en ese momento formaba parte del Virreinato de Nueva Granada. Fue un destacado líder militar y político en la lucha por la independencia de varios países sudamericanos, incluyendo Venezuela, Colombia, Ecuador, Perú y Bolivia, del dominio español.

Uno de sus logros más significativos fue la campaña que lideró en 1819, que culminó en la batalla de Boyacá, un enfrentamiento decisivo para la independencia de 
---
P: ¿Qué importancia tuvo la batalla de Boyacá en la carrera de 

## 7. RAGAS: faithfulness, answer_relevancy, context_precision

`ground_truth` son referencias cortas manuales para **context_precision**; cámbialas si modificas las preguntas.

En stderr puede aparecer *"LLM returned 1 generations instead of requested 3"*: es aviso interno de RAGAS, no un fallo. La tabla con `faithfulness`, `answer_relevancy` y `context_precision` es el resultado útil.


In [7]:
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
except ImportError:
    from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision

refs = [
    "Simón Bolívar fue un militar y político venezolano nacido en Caracas en 1783, conocido como El Libertador por liderar la independencia de varias naciones sudamericanas del dominio español.",
    "La batalla de Boyacá en 1819 fue decisiva para la independencia de la Nueva Granada y consolidó la posición militar de Bolívar en la campaña libertadora.",
]

eval_ds = Dataset.from_dict(
    {
        "question": [r["question"] for r in registros],
        "answer": [r["answer"] for r in registros],
        "contexts": [r["contexts"] for r in registros],
        "ground_truth": refs[: len(registros)],
    }
)

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

scores = evaluate(
    eval_ds,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
    ],
    llm=ragas_llm,
    embeddings=ragas_emb,
)
print(scores.to_pandas())


C:\Users\Admin\AppData\Local\Temp\ipykernel_13740\854048436.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\Admin\AppData\Local\Temp\ipykernel_13740\854048436.py:8: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
C:\Users\Admin\AppData\Local\Temp\ipykernel_13740\854048436.py:8: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextP

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


                                          user_input  \
0                           ¿Quién es Simón Bolívar?   
1  ¿Qué importancia tuvo la batalla de Boyacá en ...   

                                  retrieved_contexts  \
0  [Simón Bolívar nació el 24 de julio de 1783 en...   
1  [En 1819 lideró la campaña que culminó en la b...   

                                            response  \
0  Simón Bolívar, conocido como "El Libertador", ...   
1  La batalla de Boyacá, que tuvo lugar el 7 de a...   

                                           reference  faithfulness  \
0  Simón Bolívar fue un militar y político venezo...      0.818182   
1  La batalla de Boyacá en 1819 fue decisiva para...      0.916667   

   answer_relevancy  context_precision  
0          0.803341                1.0  
1          0.884975                1.0  
